In [ ]:
import openpyxl
import subprocess
import datetime
import time
import pyinputplus as pyip

# function to get financial report fom the bookings excel sheeet
def trainBookings(filepath):
    bookings = []
    try:
        workbook = openpyxl.load_workbook(filepath)
        sheet = workbook.active
        
        for row in sheet.iter_rows(values_only=True, min_row=2):
            if None in row[:5]:
                continue
            
            booking = {
                'TicketID': row[0],
                'PassengerName': row[1],
                'Route': row[2],
                'Class': row[3],
                'Price': row[4],
                'Status': row[5],
                'Date': row[6],
                'DiscountApplied': row[7],
                'LoyaltyMember': row[8],
            }
            bookings.append(booking)
        return bookings
    except FileNotFoundError:
        print(f'Error!!, File not found at {filepath} make sure filepath is accurate.')
        return []
    except Exception as e:
        print(f'Error occured while loading file:', e)
        return []

def summarizeBookings(bookings):
    totalTickets  = 0
    totalRevenue  = 0
    totalDiscount = 0
    cancellations = 0
    loyaltyMembers = 0
    nonLoyaltyMembers = 0
    routeSummary = {}
    totalClass = {}
    confirmedBookings = []
    cancelledBookings = []

    i = 1
    for booking in bookings:

        price = float(booking['Price'])
        discountPercentage = float(booking['DiscountApplied'])
        discountAmount = (discountPercentage/100) * price
        totalPrice = price - discountAmount
        
        status = booking['Status'].strip().lower()
        route = booking['Route']
        loyalty = booking['LoyaltyMember'].strip().lower()
        ticketClass = booking['Class'].strip()
        passenger = booking['PassengerName']
        date = booking ['Date']
        
        if status == 'confirmed':
            totalTickets += 1
            totalRevenue += totalPrice
            totalDiscount += discountAmount
            
            if ticketClass in totalClass:
                totalClass[ticketClass] += 1
            else:
                totalClass[ticketClass] = 1
            
            if route in routeSummary:
                routeSummary[route]['confirmed'] += 1
            else:
                routeSummary[route] = {'confirmed': 1, 'cancelled' : 0}
            
            if loyalty == 'yes':
                loyaltyMembers += 1
            elif loyalty == 'no':
                nonLoyaltyMembers += 1
            
            confirmedBookings.append(f'{passenger} | {route} | {ticketClass} | ${totalPrice:.2f} | {date} | Loyalty: {loyalty.capitalize()}')
            
        elif status == 'cancelled':
            cancellations += 1
            
            if route in routeSummary:
                routeSummary[route]['cancelled'] += 1
            else:
                routeSummary[route] = {'confirmed': 0, 'cancelled' : 1}
                
            cancelledBookings.append(f'{passenger} | {route} | {ticketClass} | {date}')
            
    return (totalTickets, totalRevenue, totalDiscount, cancellations, loyaltyMembers, nonLoyaltyMembers, routeSummary, totalClass, confirmedBookings, cancelledBookings)

def summary (filepath, totalTickets, totalRevenue, totalDiscount, cancellations, loyaltyMembers, nonLoyaltyMembers, routeSummary, totalClass, confirmedBookings, cancelledBookings):

    timeStamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    with open('bookingSummary.txt', mode = 'w') as file:
        file.write(f'Report generated on: {timeStamp}\n\n')
        file.write('TRAIN BOOKINGS SUMMARY\n\n')
        
        file.write(f'Total Confirmed Tickets: {totalTickets}\n')
        file.write(f'Total Revenue: ${totalRevenue:.2f}\n')
        file.write(f'Total Discount Given: ${totalDiscount:.2f}\n')
        file.write(f'Total Cancellations: {cancellations}\n')
        file.write(f'Loyalty Members: {loyaltyMembers}\n')
        file.write(f'Non-loyalty Members: {nonLoyaltyMembers}\n\n')

        file.write('TICKETS PURCHASED BY CLASS\n')
        for ticketClass in totalClass:
            file.write(f'{ticketClass} class: {totalClass[ticketClass]} ticket(s)\n')
        file.write('\n')
        
        file.write('ROUTE BOOKINGS\n')
        for route in routeSummary:
            confirmed = routeSummary[route]['confirmed']
            cancelled = routeSummary[route]['cancelled']
            file.write(f'{route} - Confirmed: {confirmed}, Cancelled: {cancelled}\n')
        file.write('\n')

        file.write('CONFIRMED TRAIN BOOKINGS\n')
        count = 1
        for detail in confirmedBookings:
            file.write(f'{count}. {detail}\n')
            count +=1 
        file.write('\n')
        
        file.write('CANCELLED TRAIN BOOKINGS\n')
        count = 1
        for detail in cancelledBookings:
            file.write(f'{count}. {detail}\n')
            count += 1


def main():
    filepath = pyip.inputFilepath(prompt= 'Enter the Train Bookings Excelfile path: ', mustExist=True)
    bookings = trainBookings(filepath)
    
    if not bookings:
        print('No Train Bookings found.')
    else:
        print('\n Processing Train Bookings.....\n')
        time.sleep(5)
        
        result = summarizeBookings(bookings)
        summary(filepath, *result)

        print('Summary saved to bookingSummary.txt')
        
        try:
            subprocess.Popen(['notepad.exe', 'bookingSummary.txt'])
            print('Opening in Notepad...')
        except Exception as e:
            print(f'Notepad could not be opened: {e}')

main()
        
    
            
                
            